# LLM Reasoning Comparison for Explainable IDS

This notebook compares multiple LLM reasoning strategies for one selected IDS traffic sample.

## SECTION 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
print(os.listdir(path))

## SECTION 2 — Install Dependencies

In [ ]:
!pip install -q transformers pandas numpy matplotlib seaborn torch

## SECTION 3 — Load Dataset

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

X_test = np.load(os.path.join(path, 'X_test.npy'))
y_test = np.load(os.path.join(path, 'y_test.npy')).reshape(-1)

if X_test.ndim == 3 and X_test.shape[-1] == 1:
    X_test = X_test[..., 0]

num_samples, num_features = X_test.shape
feature_names = [f'feature_{i}' for i in range(num_features)]
attack_classes = sorted(np.unique(y_test).tolist())

print('Number of samples:', num_samples)
print('Number of features:', num_features)
print('Feature names (first 20):', feature_names[:20])
print('Attack classes:', attack_classes)

## SECTION 4 — Select Sample

In [ ]:
sample_index = int(input('Enter traffic sample index: '))
sample_index = max(0, min(sample_index, num_samples - 1))

sample_x = X_test[sample_index]
sample_y = int(y_test[sample_index])

print('Selected sample index:', sample_index)
print('True class:', sample_y)
display(pd.DataFrame({'feature': feature_names[:20], 'value': sample_x[:20]}))

## SECTION 5 — Simulate IDS Context

In [ ]:
attack_map = {0: 'BENIGN', 1: 'DOS', 2: 'PORTSCAN', 3: 'BRUTEFORCE', 4: 'DDOS'}
attack_type = attack_map.get(sample_y, f'ATTACK_{sample_y}')

confidence = float(np.clip(0.6 + 0.25 * (sample_y > 0) + np.random.normal(0, 0.08), 0.0, 1.0))
risk_score = float(np.clip(0.55 * confidence + 0.35 * (sample_y > 0) + np.random.normal(0, 0.06), 0.0, 1.0))

top_idx = np.argsort(np.abs(sample_x))[-5:][::-1]
top_features = [feature_names[i] for i in top_idx]
memory_neighbors = [f'memory_case_{i}' for i in np.random.choice(range(100), size=3, replace=False)]
graph_neighbors = np.random.choice(['DOS', 'DDOS', 'PORTSCAN', 'INFILTRATION', 'BOTNET'], size=3, replace=False).tolist()

context = {
    'attack_type': attack_type,
    'confidence': confidence,
    'risk_score': risk_score,
    'top_features': top_features,
    'memory_neighbors': memory_neighbors,
    'graph_neighbors': graph_neighbors
}

context

## SECTION 6 — LLM Reasoning Methods

In [ ]:
from transformers import pipeline

def template_reasoning(ctx):
    return (
        f"Template reasoning: Traffic classified as {ctx['attack_type']} with confidence {ctx['confidence']:.2f} and risk {ctx['risk_score']:.2f}. "
        f"Top features: {', '.join(ctx['top_features'])}. Memory neighbors: {', '.join(ctx['memory_neighbors'])}. "
        f"Graph neighbors: {', '.join(ctx['graph_neighbors'])}. Recommended action: apply risk-tier mitigation."
    )

def cot_reasoning(ctx):
    return (
        f"Step 1: Validate class={ctx['attack_type']}, confidence={ctx['confidence']:.2f}. "
        f"Step 2: Check key signals {ctx['top_features']}. "
        f"Step 3: Compare with memory {ctx['memory_neighbors']}. "
        f"Step 4: Correlate graph links {ctx['graph_neighbors']}. "
        f"Step 5: Final risk-aware response for score {ctx['risk_score']:.2f}."
    )

def rag_reasoning(ctx):
    kb = {
        'DOS': 'DoS often causes availability degradation via request floods.',
        'DDOS': 'DDoS typically involves distributed volumetric traffic spikes.',
        'PORTSCAN': 'Port scanning indicates reconnaissance before exploitation.',
        'BRUTEFORCE': 'Bruteforce attacks involve repeated credential attempts.',
        'BENIGN': 'Benign traffic has low malicious signature overlap.'
    }
    retrieved = kb.get(ctx['attack_type'], 'Limited direct knowledge found for this class.')
    return (
        f"RAG reasoning: {retrieved} Current context shows confidence {ctx['confidence']:.2f}, risk {ctx['risk_score']:.2f}, "
        f"features {ctx['top_features']}, memory neighbors {ctx['memory_neighbors']}, graph neighbors {ctx['graph_neighbors']}."
    )

def flan_t5_reasoning(ctx, generator=None):
    prompt = (
        f"Explain IDS alert: attack={ctx['attack_type']}, confidence={ctx['confidence']:.2f}, risk={ctx['risk_score']:.2f}, "
        f"top_features={ctx['top_features']}, memory_neighbors={ctx['memory_neighbors']}, graph_neighbors={ctx['graph_neighbors']}."
    )
    if generator is None:
        return 'Flan-T5 fallback: model unavailable, use deterministic summary from context.'
    out = generator(prompt, max_length=140, do_sample=False)
    return out[0]['generated_text']

generator = None
try:
    generator = pipeline('text2text-generation', model='google/flan-t5-base')
except Exception as e:
    print('Flan-T5 unavailable, using fallback. Error:', e)

method_outputs = {}

t0 = time.perf_counter()
method_outputs['Template'] = template_reasoning(context)
template_time = time.perf_counter() - t0

t0 = time.perf_counter()
method_outputs['ChainOfThought'] = cot_reasoning(context)
cot_time = time.perf_counter() - t0

t0 = time.perf_counter()
method_outputs['RAG'] = rag_reasoning(context)
rag_time = time.perf_counter() - t0

t0 = time.perf_counter()
method_outputs['FlanT5'] = flan_t5_reasoning(context, generator=generator)
flan_time = time.perf_counter() - t0

for name, text in method_outputs.items():
    print(f'\n===== {name} =====')
    print(text)

## SECTION 7 — Comparison

In [ ]:
def clarity_score(text):
    text_l = text.lower()
    keywords = ['risk', 'confidence', 'feature', 'memory', 'graph', 'action', 'attack']
    hit = sum(1 for k in keywords if k in text_l)
    length_factor = min(len(text.split()) / 60.0, 1.0)
    return float((hit / len(keywords)) * 0.7 + length_factor * 0.3)

runtime_map = {
    'Template': template_time,
    'ChainOfThought': cot_time,
    'RAG': rag_time,
    'FlanT5': flan_time
}

rows = []
for method, text in method_outputs.items():
    rows.append({
        'Method': method,
        'ExplanationLength': len(text.split()),
        'RuntimeSec': runtime_map[method],
        'ReasoningClarity': clarity_score(text)
    })

comparison_df = pd.DataFrame(rows).sort_values(by=['ReasoningClarity', 'RuntimeSec'], ascending=[False, True]).reset_index(drop=True)
comparison_df

## SECTION 8 — Visualization

In [ ]:
sns.set(style='whitegrid')
plt.figure(figsize=(8, 4))
sns.barplot(data=comparison_df, x='Method', y='RuntimeSec', palette='Blues_d')
plt.title('Runtime Comparison of LLM Reasoning Methods')
plt.ylabel('Runtime (sec)')
plt.tight_layout()
plt.show()

## SECTION 9 — Print Best Method

In [ ]:
# Composite score: prioritize clarity, then penalize runtime
rt_norm = comparison_df['RuntimeSec'] / max(comparison_df['RuntimeSec'].max(), 1e-8)
comparison_df['CompositeScore'] = comparison_df['ReasoningClarity'] - 0.2 * rt_norm
best_row = comparison_df.sort_values(by='CompositeScore', ascending=False).iloc[0]

print('Best reasoning strategy:', best_row['Method'])
print(best_row.to_string())

comparison_df